# Pipeline 2: Multi-Hop RAG
### Claim-Verified Multi-Hop RAG — DATASCI 266, Summer 2026

**Purpose:** Improve on the naive baseline by decomposing each question into sub-questions and retrieving evidence at each hop. For each HotpotQA bridge question, we:
1. **Decompose** the question into 2 sequential sub-questions using GPT-4o-mini
    - Hop 1 sub-question targets the bridge entity (e.g. 'Who directed Pixels?')
    - Hop 2 sub-question uses the hop 1 answer to retrieve the final supporting document
2. **Retrieve** top-K passages independently for each hop, augmenting the hop 2 query with the hop 1 answer
3. **Generate** intermediate answers at each hop, then produce a final answer from all collected context
4. **Score** with the same Exact Match, Token-F1, and retrieval recall metrics as Pipeline 1

**Motivation:** Pipeline 1's strict retrieval recall was only 0.463. In 49.1% of questions the system retrieved one gold paragraph but missed the other. Single-pass retrieval cannot find the second document because the bridge entity needed to identify it is only discoverable after reading the first. Multi-hop addresses this by chaining retrieval steps.

**Shared infrastructure:** This notebook loads the corpus and FAISS index saved to Google Drive by Pipeline 1. The same hybrid retriever (BM25 + FAISS, RRF fusion, K=10) and evaluation set are used to ensure a fair comparison.

**Dataset:** HotpotQA distractor setting — bridge questions only (5,918 validation examples)  
**Retrieval:** Hybrid BM25 + FAISS, K=10 (config locked in from Pipeline 1 ablation)  
**LLM:** `gpt-4o-mini` for decomposition, hop generation, and final answer  
**Metrics:** Exact Match, Token-F1, Retrieval Recall@K (soft/strict), per-hop recall, abstention rate

In [2]:
# Install all dependencies
# Same packages as Pipeline 1 — no new installs needed for multi-hop.
# rank_bm25          : fast BM25 implementation
# sentence-transformers : pretrained bi-encoder for dense embeddings
# faiss-cpu          : Facebook's vector similarity search library
# openai             : GPT-4o-mini for decomposition and generation
# tqdm               : progress bars
%pip install rank_bm25 sentence-transformers faiss-cpu openai tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 51.5 MB/s eta 0:00:00


In [3]:
import time
import os
import re
import string
import json
import pickle
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI
from google.colab import drive, userdata

In [4]:
# OpenAI API key
# In Colab: store your key via Secrets as OPENAI_API_KEY

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

## 1. Load Corpus and Indices from Google Drive

Pipeline 1 saved the corpus and FAISS index to `MyDrive/datasci266_rag/` after building them.
Loading from Drive here means we skip the FAISS encoding step entirely.

**Files loaded:**
- `corpus.pkl` — `corpus_docs`, `corpus_titles`, `val_gold_indices` (54,391 unique paragraphs + gold index mapping)
- `faiss.index` — the pre-built FAISS flat index (54,391 vectors)
- `naive_rag_full_results.json` — Pipeline 1 final results for comparison at the end

**BM25** is rebuilt here (~30s) rather than saved, since serializing the full tokenized corpus is large and slow to load.

In [5]:
# Load corpus + FAISS index from Google Drive

# Mount Google Drive to access files saved by Pipeline 1
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"

# Load the serialized corpus dictionary saved by Pipeline 1
with open(f"{SAVE_DIR}corpus.pkl", "rb") as f:
    saved = pickle.load(f)

# corpus_docs[i] = full text of paragraph i
# corpus_titles[i] = Wikipedia article title of paragraph i
# val_gold_indices[j] = set of corpus indices that are gold paragraphs for val example j
corpus_docs = saved["docs"]
corpus_titles = saved["titles"]
val_gold_indices = saved["val_gold_indices"]

# Load the pre-built FAISS flat inner-product index (cosine similarity on normalized vectors)
index = faiss.read_index(f"{SAVE_DIR}faiss.index")

print(f"Corpus loaded:  {len(corpus_docs):,} unique paragraphs")
print(f"FAISS index:    {index.ntotal:,} vectors")
print(f"Gold mappings:  {len(val_gold_indices):,} examples")

Mounted at /content/drive
Corpus loaded:  54,391 unique paragraphs
FAISS index:    54,391 vectors
Gold mappings:  5,918 examples


In [6]:
# Rebuild BM25 index (~30 seconds)
# BM25 is not saved to disk because serializing the full tokenized corpus (~54k docs)
# is slower to pickle/unpickle than simply rebuilding it from the loaded corpus_docs.
print("Building BM25 index...")

# Convert each paragraph to lowercase tokens — matches how queries will be tokenized
tokenized_corpus = [doc.lower().split() for doc in corpus_docs]

# Initialize BM25Okapi: computes term frequencies and document lengths across the corpus
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready.")

Building BM25 index...
BM25 index ready.


In [7]:
# Load sentence-transformer (for query encoding only)
# We only need the embedder to encode incoming queries at search time
# the corpus embeddings are already stored in the FAISS index from Pipeline 1.

EMBED_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Embedder loaded: {EMBED_MODEL}")

# Load Pipeline 1 baseline results for final comparison
# Used to print a side-by-side table at the end showing improvement over the baseline
with open(f"{SAVE_DIR}naive_rag_full_results.json") as f:
    baseline_output = json.load(f)
baseline_metrics = baseline_output["metrics"]
BEST_MODE  = baseline_output["best_mode"]   # Inherit the locked-in config from Pipeline 1
BEST_TOP_K = baseline_output["best_top_k"]
print(f"\nBaseline config loaded: mode={BEST_MODE!r}, top_k={BEST_TOP_K}")
print(f"Baseline EM: {baseline_metrics['EM']:.4f} | F1: {baseline_metrics['F1']:.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded: all-MiniLM-L6-v2

Baseline config loaded: mode='hybrid', top_k=10
Baseline EM: 0.3403 | F1: 0.4459


## 2. Retrieval Functions

These functions are identical to Pipeline 1, the retrieval layer is held fixed across all three pipelines
so that any difference in results is attributable to the pipeline architecture, not retrieval.

- `bm25_retrieve` — sparse keyword search
- `dense_retrieve` — semantic vector search via FAISS
- `reciprocal_rank_fusion` — fuses both ranked lists using RRF ($k_{RRF}=60$)
- `retrieve` — unified interface: pass `mode='bm25'`, `'dense'`, or `'hybrid'`

In [8]:
# Hybrid Retrieval via Reciprocal Rank Fusion (RRF)
# Identical to Pipeline 1 — retrieval layer is held fixed across all pipelines.

def bm25_retrieve(query: str, top_n: int = 100) -> list[int]:
    """Return top_n corpus indices ranked by BM25 score."""
    # Preprocess query text exactly like the corpus tokenization step
    tokens = query.lower().split()
    # Calculate BM25 matching scores for all documents in the index
    scores = bm25.get_scores(tokens)
    # Sort indices based on score (argsort yields low-to-high, [::-1] reverses it to high-to-low)
    ranked = np.argsort(scores)[::-1]
    # Slice out the requested number of top candidates and convert to a standard Python list
    return ranked[:top_n].tolist()


def dense_retrieve(query: str, top_n: int = 100) -> list[int]:
    """Return top_n corpus indices ranked by cosine similarity (FAISS)."""
    # Convert query into a normalized 384-dimensional vector embedding matching the corpus format
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    # Run a k-Nearest Neighbors vector search inside the FAISS index (ignores scores, returns ranks)
    _, indices = index.search(q_emb, top_n)
    # Extract the array from the batch wrapper row and convert to a standard list
    return indices[0].tolist()


def reciprocal_rank_fusion(ranked_lists: list[list[int]], k: int = 60) -> list[int]:
    """
    Combine multiple ranked lists with RRF.

    For each document d appearing at rank r in a list,
    its RRF score = sum over lists of  1 / (k + r).
    k=60 is the standard default (Cormack et al., 2009).
    Returns document indices sorted by descending RRF score.
    """
    # Key = Document Index (int), Value = Accumulative RRF Score (float)
    scores: dict[int, float] = {}
    # Iterate through each system's ranked output list (BM25 list, then FAISS list)
    for ranked in ranked_lists:
        # Loop through document IDs while maintaining the 1-based rank position
        for rank, doc_idx in enumerate(ranked, start=1):
            # Add the reciprocal rank score to the document's total, initializing at 0.0 if new
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    # Sort the dictionary keys (doc IDs) based on their assigned RRF values in descending order
    return sorted(scores, key=scores.__getitem__, reverse=True)


def hybrid_retrieve(query: str, top_k: int = 10, candidate_n: int = 100) -> list[int]:
    """
    Full hybrid retrieval pipeline.

    1. BM25 retrieves `candidate_n` candidates.
    2. Dense retrieves `candidate_n` candidates.
    3. RRF merges and re-ranks both lists.
    4. Return the top_k corpus indices.

    `candidate_n` controls the recall/speed trade-off for each sub-retriever
    before fusion; 100 is a reasonable default for a ~54k paragraph corpus.
    Note: default top_k updated to 10 (selected via ablation in Pipeline 1).
    """
    # 1. Fetch top keyword matches
    bm25_results  = bm25_retrieve(query, top_n=candidate_n)
    # 2. Fetch top vector semantic matches
    dense_results = dense_retrieve(query, top_n=candidate_n)
    # 3. Intersect, score, and re-rank the union of both candidates lists using RRF math
    fused = reciprocal_rank_fusion([bm25_results, dense_results])
    # 4. Return the absolute best final matches requested by downstream application
    return fused[:top_k]


# Retrieval mode switcher
# Single entry point so the pipeline doesn't need to know which retriever is active.
# Pass mode="bm25", "dense", or "hybrid" to swap between them with no other changes.
def retrieve(query: str, top_k: int = 10, mode: str = "hybrid") -> list[int]:
    """
    Unified retrieval interface.

    mode="bm25"   — sparse lexical only (good for entity-name heavy questions)
    mode="dense"  — semantic only (good for paraphrase / synonym heavy questions)
    mode="hybrid" — RRF fusion of both (generally best; professor's recommendation)
    Note: default top_k updated to 10 (selected via ablation in Pipeline 1).
    """
    if mode == "bm25":
        return bm25_retrieve(query, top_n=top_k)
    elif mode == "dense":
        return dense_retrieve(query, top_n=top_k)
    elif mode == "hybrid":
        return hybrid_retrieve(query, top_k=top_k)
    else:
        raise ValueError(f"Unknown retrieval mode: {mode!r}. Choose 'bm25', 'dense', or 'hybrid'.")


def build_context_string(doc_indices: list[int]) -> str:
    """Format retrieved paragraphs into a numbered context block for the prompt."""
    parts = []
    # Loop over document indices, formatting them into an easy-to-read reference block for the LLM
    for i, idx in enumerate(doc_indices, 1):
        parts.append(f"[{i}] {corpus_titles[idx]}\n{corpus_docs[idx]}")
    # Join documents together with double line breaks for distinct structural spacing
    return "\n\n".join(parts)

## 3. Evaluation Metrics

Same metric functions as Pipeline 1, reproduced here so this notebook is self-contained.
All normalization follows the official HotpotQA/SQuAD evaluation protocol.

**New metric for Pipeline 2:**
- **Per-hop recall:** did hop 1's retrieval find at least one gold paragraph? Did hop 2's? This lets us
  see whether chaining actually improved retrieval of the second supporting document.

In [9]:
# Normalization (matches official HotpotQA/SQuAD eval script)

def normalize_answer(s: str) -> str:
    """Lowercase, remove punctuation, articles, and extra whitespace."""
    # Uses regex to replace standalone articles ('a', 'an', 'the') with a space
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    # Strips out duplicate internal spaces, tabs, and newlines
    def white_space_fix(text):
        return " ".join(text.split())

    # Drops all punctuation characters to avoid penalizing missing commas or periods
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    # Standardizes the string by running all cleaning steps in sequence on the lowercase text
    return white_space_fix(remove_articles(remove_punc(s.lower())))

# Returns 1 for an identical cleaned string match, or 0 if they differ
def exact_match(prediction: str, gold: str) -> int:
    """1 if the normalized prediction exactly matches the normalized gold answer."""
    return int(normalize_answer(prediction) == normalize_answer(gold))

# Break down the cleaned prediction and ground-truth strings into lists of words
def token_f1(prediction: str, gold: str) -> float:
    """
    Token-level F1 score.
    Measures partial credit — useful when answers are multi-word phrases
    and the model gets some tokens right.
    """
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    # Finds overlapping tokens by computing the intersection of word frequency counts
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    # Quick exit if there is absolutely no shared vocabulary between the strings
    if num_same == 0:
        return 0.0

    # Calculate Precision (how many predicted words are right) and Recall (how many gold words were caught)
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(gold_tokens)
    # Calculate the harmonic mean of precision and recall
    return 2 * precision * recall / (precision + recall)

# Checks if there is any intersection between the retrieved candidates and the target set
def retrieval_recall_at_k(retrieved_indices: list[int], gold_indices: set[int]) -> int:
    """
    1 if any gold paragraph appears in the retrieved set, else 0.
    'Soft' version — at least one supporting paragraph was found.
    Both gold paragraphs needed for a strict version; we track both.
    """
    return int(len(set(retrieved_indices) & gold_indices) > 0)

# Returns 1 only if every single necessary supporting document is present in your retrieval output
def both_gold_retrieved(retrieved_indices: list[int], gold_indices: set[int]) -> int:
    """1 if ALL gold paragraphs appear in the retrieved set (strict recall)."""
    return int(gold_indices.issubset(set(retrieved_indices)))


def summarize_results(results: list[dict]) -> dict:
    """Aggregate result records into mean metrics."""
    # Count total examples to use as denominator for all averages
    n = len(results)
    metrics = {
        "n":                 n,
        "EM":                sum(r["em"]            for r in results) / n,
        "F1":                sum(r["f1"]            for r in results) / n,
        "Recall@k (soft)":   sum(r["recall_soft"]   for r in results) / n,
        "Recall@k (strict)": sum(r["recall_strict"]  for r in results) / n,
        "Abstention rate":   sum(r["abstained"]      for r in results) / n,
        # New metrics for Pipeline 2: per-hop recall tracks which hop found gold paragraphs
        "Hop1 recall":       sum(r["hop1_recall"]    for r in results) / n,
        "Hop2 recall":       sum(r["hop2_recall"]    for r in results) / n,
    }
    return metrics

## 4. Multi-Hop Pipeline Functions

This section defines the three components that make up the multi-hop pipeline:

### 4.1 Question Decomposition
GPT-4o-mini is prompted to split the original question into two sequential sub-questions:
- **Q1** targets the bridge entity: the intermediate fact needed to answer the question
- **Q2** is phrased to use the Q1 answer to retrieve and identify the final answer

### 4.2 Hop Execution
Each hop retrieves top-K passages for its sub-question and generates an intermediate answer.
For hop 2, the Q1 answer is **appended to the query** before retrieval, this is what allows the
retriever to find the second gold document that single-pass retrieval misses.

### 4.3 Final Answer Generation
The final answer is generated from the original question plus all hop contexts and intermediate answers,
giving the model the full reasoning chain rather than just the last hop's context.

In [10]:
# Prompt for question decomposition
# We ask GPT-4o-mini to produce exactly 2 sub-questions in a parseable format.
# The prompt is kept concise so the model doesn't over-explain or add caveats.
DECOMPOSE_SYSTEM_PROMPT = """You are a question decomposition assistant for multi-hop reasoning.
Given a bridge question that requires two Wikipedia documents to answer, decompose it into exactly 2 sub-questions.

Q1 should identify the bridge entity (the intermediate fact).
Q2 should be answerable once Q1 is resolved.

Respond with exactly two lines and no other text:
Q1: <first sub-question>
Q2: <second sub-question>"""


def decompose_question(question: str) -> list[str]:
    """
    Use GPT-4o-mini to decompose a bridge question into 2 sub-questions.

    Returns a list of 2 strings: [q1, q2].
    Falls back to [question, question] if parsing fails, so the pipeline
    degrades gracefully rather than crashing on malformed responses.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": DECOMPOSE_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Question: {question}"},
        ],
        temperature=0,
        max_tokens=128,
    )
    text = response.choices[0].message.content.strip()

    # Parse the structured Q1:/Q2: response format
    sub_questions = []
    for line in text.split("\n"):
        line = line.strip()
        # Accept Q1:/Q2: prefix or fallback numbered list format
        for prefix in ["Q1:", "Q2:", "1.", "2."]:
            if line.startswith(prefix):
                sub_questions.append(line[len(prefix):].strip())
                break

    # Graceful fallback: if we couldn't parse 2 sub-questions, use the original question
    # This ensures the pipeline always produces an answer even if decomposition fails
    if len(sub_questions) < 2:
        return [question, question]

    return sub_questions[:2]

In [11]:
# API retry wrapper
# Wraps any API call with exponential backoff on rate limit errors.
# This prevents a single rate limit blip from crashing a multi-hour run.
# Max wait before giving up: 10 + 20 + 40 + 80 + 160 = 310 seconds (~5 min).
from openai import RateLimitError
def call_with_retry(fn, max_retries: int = 5):
    """
    Call fn() and retry with exponential backoff if a RateLimitError is raised.
    Raises the error if all retries are exhausted.
    """
    for attempt in range(max_retries):
        try:
            return fn()
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait_seconds = 10 * (2 ** attempt)  # 10s, 20s, 40s, 80s, 160s
            print(f"\nRate limit hit. Waiting {wait_seconds}s (attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait_seconds)

In [12]:
# Prompt template
# We keep the prompt minimal and consistent across all three pipelines so that
# differences in results reflect the pipeline architecture, not prompt engineering.
# Force strict grounding and a standard fallback string to prevent LLM hallucinations
SYSTEM_PROMPT = """You are a precise question-answering assistant.
Answer the question using ONLY the provided context passages.
Give a short, direct answer (a name, date, number, or brief phrase).
If the context does not contain enough information to answer, respond with exactly: UNANSWERABLE"""
def generate_answer(question: str, context: str, model: str = "gpt-4o-mini") -> str:
    """
    Call GPT-4o-mini with the question and retrieved context.
    temperature=0 for deterministic outputs — important for reproducibility
    across evaluation runs.
    """
    user_message = f"""Context passages:
{context}
Question: {question}
Answer:"""
    # Execute the API call to OpenAI's completion endpoint
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,    # Eliminates randomness to make sure evaluation results are perfectly replicable
        max_tokens=64,    # Capping length forces conciseness and cuts API token costs
    )
    # Extract the resulting text string and clean off any leading/trailing whitespace noise
    return response.choices[0].message.content.strip()
# Full multi-hop RAG pipeline
def run_multihop_rag(
    examples,
    gold_indices_list: list[set[int]],
    top_k: int = 10,
    retrieval_mode: str = "hybrid",
    verbose: bool = False,
) -> list[dict]:
    """
    Run the multi-hop RAG pipeline on a list of HotpotQA bridge examples.
    Args:
      retrieval_mode: "bm25", "dense", or "hybrid" — controls which retriever is used.
      top_k: number of passages to retrieve per hop.
    Returns a list of result dicts, one per example, with fields:
      - question, gold_answer, predicted_answer
      - sub_questions, hop1_answer, hop2_answer
      - retrieved_hop1, retrieved_hop2, em, f1
      - recall_soft, recall_strict, hop1_recall, hop2_recall
      - abstained (model said UNANSWERABLE)
    """
    results = []
    # Iterate through each sample query using a progress bar for time tracking
    for i, ex in enumerate(tqdm(examples, desc=f"multi-hop RAG [{retrieval_mode}, k={top_k}]")):
        question  = ex["question"]
        gold      = ex["answer"]
        gold_idxs = gold_indices_list[i]
        # Step 1: Decompose the original question into 2 sequential sub-questions
        sub_questions = call_with_retry(lambda: decompose_question(question))
        q1, q2 = sub_questions[0], sub_questions[1]
        # Step 2: Hop 1 — retrieve passages for q1 and generate the intermediate bridge answer
        # q1 targets the bridge entity (e.g. "Who directed Pixels?")
        retrieved_1 = retrieve(q1, top_k=top_k, mode=retrieval_mode)
        context_1   = build_context_string(retrieved_1)
        answer_1    = call_with_retry(lambda: generate_answer(q1, context_1))
        # Step 3: Hop 2 — augment q2 with the hop 1 answer to steer retrieval toward the second gold document
        # Appending answer_1 gives the retriever the bridge entity it needs — this is the key difference from single-pass
        # If hop 1 was unanswerable, fall back to q2 alone to avoid compounding a failed retrieval
        augmented_q2 = f"{q2} {answer_1}" if "UNANSWERABLE" not in answer_1.upper() else q2
        retrieved_2  = retrieve(augmented_q2, top_k=top_k, mode=retrieval_mode)
        context_2    = build_context_string(retrieved_2)
        answer_2     = call_with_retry(lambda: generate_answer(q2, context_2))
        # Step 4: Final answer — merge all retrieved passages and prepend the intermediate reasoning chain
        # De-duplicate passages across both hops so the LLM doesn't see repeated context
        all_retrieved = list(dict.fromkeys(retrieved_1 + retrieved_2))[:top_k * 2]
        final_context = build_context_string(all_retrieved)
        # Prepend intermediate findings so the LLM sees the full reasoning chain before the passages
        final_context_with_reasoning = (
            f"Intermediate findings:\n"
            f"- {q1} → {answer_1}\n"
            f"- {q2} → {answer_2}\n\n"
            f"Supporting passages:\n{final_context}"
        )
        predicted = call_with_retry(lambda: generate_answer(question, final_context_with_reasoning))
        # Step 5: Score accuracy and retrieval effectiveness against standard QA metrics
        em = exact_match(predicted, gold)
        f1 = token_f1(predicted, gold)
        # Soft/strict recall measured across the union of both hops' retrieved passages
        recall_soft = retrieval_recall_at_k(all_retrieved, gold_idxs)
        recall_strict = both_gold_retrieved(all_retrieved, gold_idxs)
        # Per-hop recall: diagnoses whether hop 1 or hop 2 was the retrieval bottleneck
        hop1_recall = retrieval_recall_at_k(retrieved_1, gold_idxs)
        hop2_recall = retrieval_recall_at_k(retrieved_2, gold_idxs)
        # Track whether the LLM correctly identified a lack of information instead of hallucinating
        abstained = int("UNANSWERABLE" in predicted.upper())
        # Consolidate metrics and text info into a structured evaluation logging dictionary
        record = dict(
            question=question,
            gold_answer=gold,
            predicted_answer=predicted,
            sub_questions=[q1, q2],
            hop1_answer=answer_1,
            hop2_answer=answer_2,
            retrieved_hop1=retrieved_1,
            retrieved_hop2=retrieved_2,
            em=em,
            f1=f1,
            recall_soft=recall_soft,
            recall_strict=recall_strict,
            hop1_recall=hop1_recall,
            hop2_recall=hop2_recall,
            abstained=abstained,
        )
        results.append(record)
        # Print detailed individual query summaries dynamically if verbose flag is set
        if verbose:
            status = "✓" if em else "✗"
            print(f"{status} Q:  {question[:70]}")
            print(f"   Q1: {q1[:60]} → {answer_1!r}")
            print(f"   Q2: {q2[:60]} → {answer_2!r}")
            print(f"   Gold: {gold!r}  |  Pred: {predicted!r}  |  F1: {f1:.2f}")
            print()
    return results

## 5. Run the Multi-Hop Pipeline

Same three-stage run as Pipeline 1:

1. **5.1 Smoke test** (10 examples, verbose) — confirm decomposition, retrieval, and generation all work end-to-end
2. **5.2 Full evaluation run** — all 2,000 subset bridge questions, same set as Pipeline 1
3. **5.3 Comparison** — side-by-side table against Pipeline 1 baseline

**No ablation needed here** — retrieval config (hybrid, K=10) is inherited from Pipeline 1.

**Cost estimate:** Each question now makes 3 API calls (decompose + 2 hop answers) plus 1 for the final answer = 4 calls per example.

### 5.1 Smoke Test

Run on 10 examples with `verbose=True` to inspect the full reasoning chain for each question:
the two sub-questions, both intermediate answers, and the final predicted answer.
This confirms decomposition is producing sensible sub-questions before committing to the full run.

In [12]:
# Smoke test on 10 examples
# Run this first to confirm the pipeline works end-to-end before spending API budget.

# Load the validation set to get example objects
from datasets import load_dataset
print("Loading HotpotQA (bridge questions only)...")
dataset = load_dataset("hotpotqa/hotpot_qa", "distractor")
val_data = dataset["validation"].filter(lambda x: x["type"] == "bridge")
print(f"Bridge questions: {len(val_data):,}")

SMOKE_N = 10
smoke_examples = [val_data[i] for i in range(SMOKE_N)]
smoke_gold_indices = val_gold_indices[:SMOKE_N]

smoke_results = run_multihop_rag(
    smoke_examples,
    smoke_gold_indices,
    top_k=BEST_TOP_K,
    retrieval_mode=BEST_MODE,
    verbose=True,
)

smoke_metrics = summarize_results(smoke_results)
print("── Smoke test results ──")
for k, v in smoke_metrics.items():
    print(f"  {k}: {v}" if k == 'n' else f"  {k:<25}: {v:.3f}")

Loading HotpotQA (bridge questions only)...


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7405 [00:00<?, ? examples/s]

Bridge questions: 5,918


multi-hop RAG [hybrid, k=10]:   0%|          | 0/10 [00:00<?, ?it/s]

✗ Q:  What government position was held by the woman who portrayed Corliss A
   Q1: Who portrayed Corliss Archer in the film Kiss and Tell? → 'Shirley Temple'
   Q2: What government position did she hold? → 'UNANSWERABLE'
   Gold: 'Chief of Protocol'  |  Pred: 'United States ambassador to Ghana and to Czechoslovakia'  |  F1: 0.00

✗ Q:  What science fantasy young adult series, told in first person, has a s
   Q1: What is the name of the science fantasy young adult series t → 'UNANSWERABLE'
   Q2: Who is the author of this science fantasy young adult series → 'Katherine Applegate'
   Gold: 'Animorphs'  |  Pred: 'UNANSWERABLE'  |  F1: 0.00

✓ Q:  The director of the romantic comedy "Big Stone Gap" is based in what N
   Q1: Who is the director of the romantic comedy "Big Stone Gap"? → 'Adriana Trigiani'
   Q2: In which New York city is this director based? → 'Greenwich Village, New York City'
   Gold: 'Greenwich Village, New York City'  |  Pred: 'Greenwich Village, New York City'  |  F1: 

### 5.2 Full Evaluation Run

Runs the multi-hop pipeline on the shared 2,000-example eval set (loaded from eval_indices_shared.npy) that was verified as representative of the full validation set in Pipeline 1.

Checkpointing: results are saved to Drive every 100 examples. If the run is interrupted, re-run this cell and it will automatically resume from the last checkpoint.

Decomposition reuse: sub-questions [q1, q2] are saved to decompositions.json on Drive. Pipeline 3 loads these directly, eliminating 2,000 redundant API calls and ensuring both pipelines use identical decompositions.

In [13]:
# Full run on the shared 2,000-example eval set
# Loads the shared eval indices saved by Pipeline 1's subset verification cell.
# Includes checkpointing (save every 100 examples) and auto-resume on restart.

from datasets import load_dataset

print("Loading HotpotQA bridge questions...")
dataset  = load_dataset("hotpotqa/hotpot_qa", "distractor")
val_data = dataset["validation"].filter(lambda x: x["type"] == "bridge")
print(f"Bridge questions available: {len(val_data):,}")

# Load the shared eval indices locked in by Pipeline 1's subset verification
eval_indices_shared = np.load(f"{SAVE_DIR}eval_indices_shared.npy")
eval_examples  = [val_data[int(i)] for i in eval_indices_shared]
eval_gold_indices_list = [val_gold_indices[int(i)] for i in eval_indices_shared]
print(f"Eval set loaded: {len(eval_examples):,} examples (shared indices, seed=42)")

# Resume from checkpoint if one exists
# Checkpoints save results every 100 examples so a crash only loses the last 100 at most.
CHECKPOINT_PATH = f"{SAVE_DIR}multihop_rag_checkpoint.json"

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    full_results = checkpoint["results"]
    start_idx    = len(full_results)
    print(f"\nResuming from checkpoint: {start_idx} done, {len(eval_examples) - start_idx} remaining.")
else:
    full_results = []
    start_idx    = 0
    print(f"\nStarting fresh run on {len(eval_examples):,} examples.")

print(f"Config: mode={BEST_MODE!r}, top_k={BEST_TOP_K}")
print(f"Estimated API calls remaining: {(len(eval_examples) - start_idx) * 4:,}")

# Run the evaluation pipeline with resumable checkpointing
# Tracks indices globally using start_idx to safely resume from where it left off
for i, (ex, gold_idxs) in enumerate(
    zip(eval_examples[start_idx:], eval_gold_indices_list[start_idx:]),
    start=start_idx
):
    # Extract the original question and the target ground-truth answer
    question = ex["question"]
    gold = ex["answer"]

    # Step 1: Load saved decomposition (no API call)
    # Break down the complex main question into two sequential sub-questions (q1 and q2)
    sub_questions = call_with_retry(lambda: decompose_question(question))
    q1, q2 = sub_questions[0], sub_questions[1]

    # Step 2: Hop 1
    # Retrieve relevant document passages for the first sub-question, format them,
    # and generate an initial intermediate answer.
    retrieved_1 = retrieve(q1, top_k=BEST_TOP_K, mode=BEST_MODE)
    context_1 = build_context_string(retrieved_1)
    answer_1 = call_with_retry(lambda: generate_answer(q1, context_1))

    # Step 3: Hop 2 (Conditional Augmentation)
    # If Hop 1 successfully found an answer, append that answer to the second sub-question
    # to provide necessary context for the next retrieval. Otherwise, search using q2 alone.
    augmented_q2 = f"{q2} {answer_1}" if "UNANSWERABLE" not in answer_1.upper() else q2
    retrieved_2 = retrieve(augmented_q2, top_k=BEST_TOP_K, mode=BEST_MODE)
    context_2 = build_context_string(retrieved_2)
    answer_2 = call_with_retry(lambda: generate_answer(q2, context_2))

    # Step 4: Final answer generation
    # Deduplicate and combine retrieved passages from both hops, while respecting a maximum capacity limit.
    all_retrieved = list(dict.fromkeys(retrieved_1 + retrieved_2))[:BEST_TOP_K * 2]
    final_context = build_context_string(all_retrieved)

    # Synthesize a prompt containing the step-by-step reasoning trail and all supporting documents
    final_context_with_reasoning = (
        f"Intermediate findings:\n- {q1} → {answer_1}\n- {q2} → {answer_2}\n\n"
        f"Supporting passages:\n{final_context}"
    )

    # Prompt the LLM to generate the final end-user answer using the full reasoning context
    predicted = call_with_retry(lambda: generate_answer(question, final_context_with_reasoning))

    # Step 5: Score
    em            = exact_match(predicted, gold)                      # Strict exact matching (0 or 1)
    f1            = token_f1(predicted, gold)                         # Token-level overlap F1 score
    recall_soft   = retrieval_recall_at_k(all_retrieved, gold_idxs)   # Check if *any* gold index was retrieved
    recall_strict = both_gold_retrieved(all_retrieved, gold_idxs)     # Check if *all* required gold indices were retrieved
    hop1_recall   = retrieval_recall_at_k(retrieved_1, gold_idxs)     # Document recall performance for Hop 1
    hop2_recall   = retrieval_recall_at_k(retrieved_2, gold_idxs)     # Document recall performance for Hop 2
    abstained     = int("UNANSWERABLE" in predicted.upper())          # Track if the model safely refused to answer

    # Consolidate metrics and text info into a structured evaluation logging dictionary
    record = dict(
        question=question, gold_answer=gold, predicted_answer=predicted,
        sub_questions=[q1, q2],   # ← saved for Pipeline 3 reuse
        hop1_answer=answer_1, hop2_answer=answer_2,
        retrieved_hop1=retrieved_1, retrieved_hop2=retrieved_2,
        em=em, f1=f1, recall_soft=recall_soft, recall_strict=recall_strict,
        hop1_recall=hop1_recall, hop2_recall=hop2_recall, abstained=abstained,
    )
    full_results.append(record)

    # Save checkpoint every 100 examples
    if (i + 1) % 100 == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump({"results": full_results, "completed": i + 1}, f)
        print(f"  Checkpoint saved: {i + 1}/{len(eval_examples)}")

# Final save
full_metrics = summarize_results(full_results)

print("\n══════════════════════════════════════════")
print("  Multi-Hop RAG — FINAL RESULTS")
print(f"  n = {full_metrics['n']:,} (shared eval set, seed=42)")
print(f"  retrieval_mode={BEST_MODE!r}, top_k={BEST_TOP_K}")
print("══════════════════════════════════════════")
for k, v in full_metrics.items():
    if k != "n":
        print(f"  {k:<25}: {v:.4f}")
print("══════════════════════════════════════════")

output = {
    "pipeline": "multihop_rag", "split": "shared_eval_2000_seed42",
    "n": len(full_results), "best_mode": BEST_MODE, "best_top_k": BEST_TOP_K,
    "embed_model": EMBED_MODEL, "llm": "gpt-4o-mini",
    "metrics": full_metrics, "results": full_results,
}
with open("multihop_rag_results.json", "w") as f:
    json.dump(output, f, indent=2)
with open(f"{SAVE_DIR}multihop_rag_results.json", "w") as f:
    json.dump(output, f, indent=2)

# Save decompositions for Pipeline 3 reuse
decompositions = {r["question"]: r["sub_questions"] for r in full_results}
with open(f"{SAVE_DIR}decompositions.json", "w") as f:
    json.dump(decompositions, f, indent=2)

print(f"\nResults + decompositions saved to Google Drive.")

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("Checkpoint removed (run complete).")

Loading HotpotQA bridge questions...
Bridge questions available: 5,918
Eval set loaded: 2,000 examples (shared indices, seed=42)

Resuming from checkpoint: 1900 done, 100 remaining.
Config: mode='hybrid', top_k=10
Estimated API calls remaining: 400
  Checkpoint saved: 2000/2000

══════════════════════════════════════════
  Multi-Hop RAG — FINAL RESULTS
  n = 2,000 (shared eval set, seed=42)
  retrieval_mode='hybrid', top_k=10
══════════════════════════════════════════
  EM                       : 0.4240
  F1                       : 0.5552
  Recall@k (soft)          : 0.9570
  Recall@k (strict)        : 0.6950
  Abstention rate          : 0.2400
  Hop1 recall              : 0.9110
  Hop2 recall              : 0.8700
══════════════════════════════════════════

Results + decompositions saved to Google Drive.
Checkpoint removed (run complete).


### 5.3 Comparison to Pipeline 1 Baseline

Side-by-side results for Pipeline 1 (naive single-pass) and Pipeline 2 (multi-hop).

In [14]:
# Side-by-side comparison: Pipeline 1 vs Pipeline 2

# Shared metric keys to compare across pipelines
shared_metrics = ["EM", "F1", "Recall@k (soft)", "Recall@k (strict)", "Abstention rate"]

print(f"{'Metric':<25} {'Pipeline 1':>12} {'Pipeline 2':>12} {'Delta':>10}")
print("─" * 62)
for metric in shared_metrics:
    v1 = baseline_metrics[metric]
    v2 = full_metrics[metric]
    delta = v2 - v1
    # Show + for improvements, - for regressions
    arrow = "▲" if delta > 0.001 else ("▼" if delta < -0.001 else "─")
    print(f"{metric:<25} {v1:>12.4f} {v2:>12.4f} {arrow} {delta:>+.4f}")

print()
print(f"{'Hop1 recall':<25} {'N/A':>12} {full_metrics['Hop1 recall']:>12.4f}")
print(f"{'Hop2 recall':<25} {'N/A':>12} {full_metrics['Hop2 recall']:>12.4f}")

# Error analysis
print("\n── Error analysis ──")
failures               = [r for r in full_results if r["em"] == 0]
retrieval_ok_but_wrong = [r for r in failures if r["recall_strict"] == 1]
retrieval_failed       = [r for r in failures if r["recall_soft"] == 0]
decompose_fallback     = [r for r in full_results if r["sub_questions"][0] == r["question"]]

print(f"Total failures:                          {len(failures)}")
print(f"  Both gold paras retrieved, still wrong: {len(retrieval_ok_but_wrong)} → Generation failure")
print(f"  Gold paras not retrieved at all:        {len(retrieval_failed)} → Retrieval failure")
print(f"Decomposition fallback (parse failed):   {len(decompose_fallback)}")
print()

# Print a few examples where multi-hop succeeded but baseline would have failed
print("Sample multi-hop successes (em=1, both hops retrieved something useful):")
successes = [r for r in full_results if r["em"] == 1 and r["recall_strict"] == 1]
for r in successes[:2]:
    print(f"  Q:  {r['question']}")
    print(f"  Q1: {r['sub_questions'][0]} → {r['hop1_answer']!r}")
    print(f"  Q2: {r['sub_questions'][1]} → {r['hop2_answer']!r}")
    print(f"  Gold: {r['gold_answer']!r}  |  Pred: {r['predicted_answer']!r}")
    print()

Metric                      Pipeline 1   Pipeline 2      Delta
──────────────────────────────────────────────────────────────
EM                              0.3403       0.4240 ▲ +0.0837
F1                              0.4459       0.5552 ▲ +0.1093
Recall@k (soft)                 0.9542       0.9570 ▲ +0.0028
Recall@k (strict)               0.4633       0.6950 ▲ +0.2317
Abstention rate                 0.3552       0.2400 ▼ -0.1152

Hop1 recall                        N/A       0.9110
Hop2 recall                        N/A       0.8700

── Error analysis ──
Total failures:                          1152
  Both gold paras retrieved, still wrong: 643 → Generation failure
  Gold paras not retrieved at all:        84 → Retrieval failure
Decomposition fallback (parse failed):   1

Sample multi-hop successes (em=1, both hops retrieved something useful):
  Q:  What year did a director of a North Korean cinema film get kidnapped?
  Q1: Who is the director of the North Korean cinema film that was

## 6. Post-Hoc Faithfulness Analysis (Pipeline 3 Baseline)

Pipeline 2 always propagates hop 1 answers to hop 2 without checking whether those answers are grounded in the retrieved evidence. This is the correct behavior for a multi-hop RAG baseline, but it means we cannot distinguish between three outcome types:
- **Faithful correct answers** — right answer, supported by retrieved passages
- **Unfaithful correct answers** — right answer, but not traceable to the retrieved evidence (right for the wrong reason, a faithfulness failure EM cannot detect)
- **Hallucinations** — wrong answer, not supported by retrieved passages

To measure the **hallucination compounding hypothesis** — that unsupported intermediate claims propagate into incorrect final answers, we need a faithfulness baseline for Pipeline 2. We apply the same DeBERTa NLI verifier used in Pipeline 3 **post-hoc** to the already-saved hop 1 and hop 2 answers. No API calls are made; the NLI model runs locally on the saved results.

**This analysis serves as the unverified-propagation baseline.** Pipeline 3 records these same grounding labels during its run, enabling a direct comparison between the compounding pattern when no intervention is applied (Pipeline 2) and when a verifier is used (Pipeline 3).

**Metrics computed:**
- **Per-hop verification failure rate** — fraction of hop 1 / hop 2 answers flagged as not entailed by their retrieved passages
- **Conditional EM by grounding state** — mean EM broken down by which hops were grounded:
  - *Both grounded* — hop 1 and hop 2 both supported
  - *Hop 1 only flagged* — hop 1 unsupported, hop 2 supported (subcategory of "one flagged")
  - *Hop 2 only flagged* — hop 1 supported, hop 2 unsupported (subcategory of "one flagged")
  - *Both flagged* — neither hop supported
  - If hallucination compounds, EM should decrease monotonically from both-grounded → both-flagged
- **Faithfulness failure rate** — fraction of correct final answers (EM = 1) where at least one hop was ungrounded
- **Propagation penalty** — EM difference between examples where hop 1 was grounded vs. ungrounded; quantifies the cost of propagating an unsupported intermediate answer

## Verifier Calibration and Grounding Analysis

This section runs a post-hoc verifier pipeline on saved Pipeline 2 outputs to measure faithfulness and hallucination propagation without rerunning any OpenAI generation calls.

### Objective
We want a verifier signal that is actually useful for multi-hop QA. A useful verifier should flag intermediate answers that are more likely to lead to final answer failure.

### Method Overview
1. **Verifier ablation (hop 1 only):**
   - Evaluate multiple NLI verifier models on saved hop 1 outputs.
   - Compare models using:
     - **Flag rate**
     - **Failure rate when flagged**
     - **Failure rate when not flagged**
     - **Lift = Fail|flagged − Fail|not flagged**
   - Select `BEST_VERIFIER` as the model with the highest lift.

2. **Full grounding analysis (both hops):**
   - Apply `BEST_VERIFIER` to hop 1 and hop 2 for all examples.
   - Compute faithfulness metrics:
     - **Per-hop verification failure rate**
     - **Conditional EM by grounding state** (both grounded / one flagged with hop-specific subcategories / both flagged)
     - **Faithfulness failure rate** (correct final answers with at least one ungrounded hop)
     - **Propagation penalty** (EM drop when hop 1 is ungrounded vs grounded)

### QA-to-Claim Verification Input (Updated)
Verifier inputs are built as **declarative claims** from the question-answer pair, rather than passing answer fragments alone.

- Example: `Q: On what network did The Strain premiere?` + `A: FX` → claim like `The Strain premiered on FX.`

This better matches how NLI models are trained and improves grounding interpretability for short factoid answers.

### Data and Outputs
- **Inputs loaded from Drive:**
  - `multihop_rag_results.json`
  - `corpus.pkl`
- **Output written to Drive:**
  - `multihop_rag_grounding_results.json`

The output file is the grounding-annotated baseline used for downstream comparison in the next pipeline stage.

In [1]:
# 7.1 Verifier ablation (hop 1 only)
# Clean standalone cell: loads saved results from Drive if needed.

%pip install transformers torch --quiet

import json
import pickle
import re
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

# Ensure Drive + saved data are available in fresh sessions
if "SAVE_DIR" not in dir():
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"

# Load primary generation output results if not already present in memory
if "full_results" not in dir() or full_results is None:
    with open(f"{SAVE_DIR}multihop_rag_results.json") as f:
        _saved = json.load(f)
    full_results = _saved["results"]

# Load the source corpus passages to map indices back to actual text premises
if "corpus_docs" not in dir() or corpus_docs is None:
    with open(f"{SAVE_DIR}corpus.pkl", "rb") as f:
        _corpus = pickle.load(f)
    corpus_docs = _corpus["docs"]

# Configure hardware acceleration (GPU if available, fallback to CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Per-passage verifier settings
# Limit evaluation to the top K passages to prevent long inference times
MAX_PASSAGES_VERIFY = 3
# Minimal threshold for an ENTAILMENT probability to consider a passage "supporting"
ENTAILMENT_THRESHOLD = 0.50

print(f"Running on: {DEVICE}")
print(f"Loaded results: {len(full_results):,} examples")
print(f"Loaded corpus:  {len(corpus_docs):,} passages")
print(f"Verifier aggregation: per-passage max entailment (top {MAX_PASSAGES_VERIFY}, threshold={ENTAILMENT_THRESHOLD:.2f})\n")

# Candidate models (baseline + FEVER-aligned variants)
CANDIDATE_MODELS = [
    "cross-encoder/nli-deberta-v3-base",
    "MoritzLaurer/DeBERTa-v3-base-mnli-fever-docnli-ling-2c",
    "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
]

# Transforms a Question-Answer pair into a unified, declarative sentence.
# This creates a clean 'hypothesis' string optimized for NLI classification models.
def qa_to_claim(question: str, answer: str) -> str:
    """Convert QA pair into a declarative claim for NLI."""
    q = question.strip()
    a = answer.strip()
    if not q:
        return a

    # Clean off trailing punctuation to make string merging easier
    q_no_qmark = q[:-1] if q.endswith("?") else q

    # Lightweight rewrite for common WH-leading questions to swap out leading WH-words with the generated answer string
    m = re.match(r"^(Who|What|Where|When|Which|Whose|How many|How much)\b", q_no_qmark, flags=re.I)
    if m:
        claim = re.sub(r"^(Who|What|Where|When|Which|Whose|How many|How much)\b", a, q_no_qmark, flags=re.I)
        return claim if claim.endswith(".") else claim + "."

    return f"Question: {q} Answer: {a}."

# Closure factory that binds a tokenizer, model, device, and entailment index.
# Returns a unified 'verify()' utility function for evaluating generated answers.
def make_verifier(tok, mdl, dev, ent_idx):
    def verify(question: str, answer: str, context_indices: list, max_passages: int = MAX_PASSAGES_VERIFY) -> dict:
        # Base Case 1: If the generator explicitly refused to answer, bypass NLI checking
        # Track UNANSWERABLE as its own category (not auto-supported, not auto-flagged).
        if "UNANSWERABLE" in answer.upper():
            return {
                "supported": None,
                "state": "UNANSWERABLE",
                "label": "UNANSWERABLE",
                "entailment_score": 1.0,
                "claim": qa_to_claim(question, answer),
                "aggregation": "max_entailment",
                "passage_scores": [],
                "vote_label": "UNANSWERABLE",
            }
        # Base Case 2: If no context was retrieved, the answer is inherently unsupported
        if not context_indices:
            return {
                "supported": False,
                "state": "UNSUPPORTED",
                "label": "NEUTRAL",
                "entailment_score": 0.0,
                "claim": qa_to_claim(question, answer),
                "aggregation": "max_entailment",
                "passage_scores": [],
                "vote_label": "NEUTRAL",
            }

        claim = qa_to_claim(question, answer)

        # Per-passage verification: score each passage independently, then aggregate.
        passage_scores = []
        passage_labels = []

        for idx in context_indices[:max_passages]:
            premise = corpus_docs[idx]
            inputs = tok(
                premise,
                claim,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True,
            ).to(dev)

            with torch.no_grad():
                logits = mdl(**inputs).logits

            # Map raw model logits to stable percentage probabilities
            probs = torch.softmax(logits, dim=-1)[0]
            pred_idx = probs.argmax().item()
            pred_label = mdl.config.id2label[pred_idx].upper()
            ent_score = probs[ent_idx].item()

            passage_scores.append(ent_score)
            passage_labels.append(pred_label)

        # Aggregate using max entailment score: if any passage strongly supports,
        # mark the hop as supported.
        max_ent = max(passage_scores) if passage_scores else 0.0
        is_supported = max_ent >= ENTAILMENT_THRESHOLD

        # Vote diagnostic (for analysis only)
        vote_counts = {}
        for lbl in passage_labels:
            vote_counts[lbl] = vote_counts.get(lbl, 0) + 1
        vote_label = sorted(vote_counts.items(), key=lambda x: (-x[1], x[0]))[0][0] if vote_counts else "NEUTRAL"

        return {
            "supported": is_supported,
            "state": "SUPPORTED" if is_supported else "UNSUPPORTED",
            "label": "ENTAILMENT" if is_supported else vote_label,
            "entailment_score": max_ent,
            "claim": claim,
            "aggregation": "max_entailment",
            "threshold": ENTAILMENT_THRESHOLD,
            "passage_scores": passage_scores,
            "passage_labels": passage_labels,
            "vote_label": vote_label,
        }

    return verify

# Ablation Testing Loop
ablation_results = {}

for model_name in CANDIDATE_MODELS:
    short = model_name.split("/")[-1]
    print(f"Loading: {short}...")

    # Load and initialize model architecture + vocabulary weights
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
    mdl.eval()
    mdl = mdl.to(DEVICE)

    # Locate the correct output tensor index representing the ENTAILMENT class
    lbl_map = {v.lower(): k for k, v in mdl.config.id2label.items()}
    ent_idx = lbl_map.get("entailment", 1)
    verify = make_verifier(tok, mdl, DEVICE, ent_idx)

    verifications = []
    # Run verification predictions over the entire generated dataset
    for r in tqdm(full_results, desc=f"  {short[:38]}", leave=False):
        # Handle cases where sub_questions list may be missing or unpopulated
        q1 = r["sub_questions"][0] if "sub_questions" in r and len(r["sub_questions"]) >= 1 else r["question"]
        v = verify(q1, r["hop1_answer"], r["retrieved_hop1"])
        # Capture the original exact match (EM) performance alongside new verifier flags
        verifications.append({"em": r["em"], "state": v["state"], "supported": v["supported"]})

    # Segment results into clean subsets based on NLI states
    unsupported = [v for v in verifications if v["state"] == "UNSUPPORTED"]
    supported = [v for v in verifications if v["state"] == "SUPPORTED"]
    unanswerable = [v for v in verifications if v["state"] == "UNANSWERABLE"]
    decided = supported + unsupported

    # Compute descriptive metrics and error rates
    unsupported_rate_all = len(unsupported) / len(verifications)
    unanswerable_rate = len(unanswerable) / len(verifications)

    # Error rate (1 - EM) under each verifier classification bucket
    fail_if_unsupported = sum(1 for v in unsupported if v["em"] == 0) / max(len(unsupported), 1)
    fail_if_supported = sum(1 for v in supported if v["em"] == 0) / max(len(supported), 1)
    fail_if_unanswerable = sum(1 for v in unanswerable if v["em"] == 0) / max(len(unanswerable), 1)

    # Lift computed on decided examples only (SUPPORTED vs UNSUPPORTED)
    # Lift Calculation: Difference in failure rates. Higher lift indicates the verifier
    # is successfully separating bad generation errors from correct answers.
    lift = fail_if_unsupported - fail_if_supported

    # Store computed stats for side-by-side terminal formatting
    ablation_results[model_name] = {
        "unsupported_rate_all": unsupported_rate_all,
        "unanswerable_rate": unanswerable_rate,
        "fail_if_unsupported": fail_if_unsupported,
        "fail_if_supported": fail_if_supported,
        "fail_if_unanswerable": fail_if_unanswerable,
        "decided_n": len(decided),
        "lift": lift,
    }

    print(
        f"  unsupported={unsupported_rate_all:.1%}  unanswerable={unanswerable_rate:.1%}  "
        f"fail|unsup={fail_if_unsupported:.1%}  fail|sup={fail_if_supported:.1%}  lift={lift:+.3f}"
    )

    # Clean up large tensor graph structures manually to prevent Notebook CUDA Out-Of-Memory crashes
    del mdl, tok, verify
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Final Report Printing
print("\n── Verifier ablation results (hop 1, n=2,000) ──")
print(f"{'Model':<44} {'Unsup%':>7} {'Unans%':>7} {'Fail|unsup':>11} {'Fail|sup':>9} {'Lift':>8}")
print("─" * 94)
for name, m in ablation_results.items():
    short = name.split("/")[-1][:42]
    marker = " ◄" if m["lift"] == max(v["lift"] for v in ablation_results.values()) else ""
    print(
        f"{short:<44} {m['unsupported_rate_all']:>7.1%} {m['unanswerable_rate']:>7.1%} "
        f"{m['fail_if_unsupported']:>11.1%} {m['fail_if_supported']:>9.1%} {m['lift']:>+8.3f}{marker}"
    )

BEST_VERIFIER = max(ablation_results, key=lambda k: ablation_results[k]["lift"])
print(f"\nSelected BEST_VERIFIER: {BEST_VERIFIER}")
print(
    f"Lift={ablation_results[BEST_VERIFIER]['lift']:+.3f} | "
    f"Unsupported rate={ablation_results[BEST_VERIFIER]['unsupported_rate_all']:.1%} | "
    f"UNANSWERABLE rate={ablation_results[BEST_VERIFIER]['unanswerable_rate']:.1%}"
)
print('Override manually if needed:  # BEST_VERIFIER = "<model-name>"')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on: cuda
Loaded results: 2,000 examples
Loaded corpus:  54,391 passages
Verifier aggregation: per-passage max entailment (top 3, threshold=0.50)

Loading: nli-deberta-v3-base...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  nli-deberta-v3-base:   0%|          | 0/2000 [00:00<?, ?it/s]

  unsupported=32.6%  unanswerable=21.2%  fail|unsup=52.4%  fail|sup=49.2%  lift=+0.031
Loading: DeBERTa-v3-base-mnli-fever-docnli-ling-2c...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  DeBERTa-v3-base-mnli-fever-docnli-ling:   0%|          | 0/2000 [00:00<?, ?it/s]

  unsupported=22.0%  unanswerable=21.2%  fail|unsup=48.9%  fail|sup=51.2%  lift=-0.023
Loading: DeBERTa-v3-large-mnli-fever-anli-ling-wanli...


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

  DeBERTa-v3-large-mnli-fever-anli-ling-:   0%|          | 0/2000 [00:00<?, ?it/s]

  unsupported=21.1%  unanswerable=21.2%  fail|unsup=51.2%  fail|sup=50.3%  lift=+0.009

── Verifier ablation results (hop 1, n=2,000) ──
Model                                         Unsup%  Unans%  Fail|unsup  Fail|sup     Lift
──────────────────────────────────────────────────────────────────────────────────────────────
nli-deberta-v3-base                            32.6%   21.2%       52.4%     49.2%   +0.031 ◄
DeBERTa-v3-base-mnli-fever-docnli-ling-2c      22.0%   21.2%       48.9%     51.2%   -0.023
DeBERTa-v3-large-mnli-fever-anli-ling-wanl     21.1%   21.2%       51.2%     50.3%   +0.009

Selected BEST_VERIFIER: cross-encoder/nli-deberta-v3-base
Lift=+0.031 | Unsupported rate=32.6% | UNANSWERABLE rate=21.2%
Override manually if needed:  # BEST_VERIFIER = "<model-name>"


In [2]:
# Threshold sweep for verifier calibration (no OpenAI API calls)
# Uses the verifier selected in 7.1 and sweeps entailment thresholds to find
# the best lift on hop-1 decided cases (SUPPORTED vs UNSUPPORTED).

# Install the required Hugging Face libraries for model serving and execution
%pip install transformers torch --quiet

import json
import pickle
import re
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

print("No OpenAI API calls in this cell — local NLI inference only.\n")

# Ensure Drive + saved data are available in fresh sessions
# Establish paths and connect to Google Drive if variables are not already in memory
if "SAVE_DIR" not in dir():
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"

# Load evaluation results from previous multi-hop RAG pipeline runs
if "full_results" not in dir() or full_results is None:
    with open(f"{SAVE_DIR}multihop_rag_results.json") as f:
        _saved = json.load(f)
    full_results = _saved["results"]

# Load the comprehensive corpus containing raw background documents/passages
if "corpus_docs" not in dir() or corpus_docs is None:
    with open(f"{SAVE_DIR}corpus.pkl", "rb") as f:
        _corpus = pickle.load(f)
    corpus_docs = _corpus["docs"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_PASSAGES_VERIFY = 3

# Runtime Configuration: Detect hardware and set constraints
# Require 7.1 first so model choice is explicit and reproducible.
if "BEST_VERIFIER" not in dir() or not BEST_VERIFIER:
    raise RuntimeError("BEST_VERIFIER is not set. Run '7.1 Verifier ablation' first.")

print(f"Running on: {DEVICE}")
print(f"Verifier model: {BEST_VERIFIER}")
print(f"Examples: {len(full_results):,}")
print(f"Per-passage aggregation over top {MAX_PASSAGES_VERIFY}\n")

# Reuse qa_to_claim from 7.1 if already defined.
# Converts a question-answer pair into a declarative statement (claim).
# Uses simple regex heuristics to replace question words with the answer string.
if "qa_to_claim" not in dir():
    def qa_to_claim(question: str, answer: str) -> str:
        q = question.strip()
        a = answer.strip()
        if not q:
            return a
        q_no_qmark = q[:-1] if q.endswith("?") else q
        m = re.match(r"^(Who|What|Where|When|Which|Whose|How many|How much)\b", q_no_qmark, flags=re.I)
        if m:
            claim = re.sub(r"^(Who|What|Where|When|Which|Whose|How many|How much)\b", a, q_no_qmark, flags=re.I)
            return claim if claim.endswith(".") else claim + "."
        return f"Question: {q} Answer: {a}."

# Load model once
# Load selected Sequence Classification weights and put model in evaluation mode
nli_tokenizer = AutoTokenizer.from_pretrained(BEST_VERIFIER)
nli_model = AutoModelForSequenceClassification.from_pretrained(BEST_VERIFIER)
nli_model.eval()
nli_model = nli_model.to(DEVICE)

# Dynamically map label name strings back to classification indices
label_to_idx = {v.lower(): k for k, v in nli_model.config.id2label.items()}
ENTAILMENT_IDX = label_to_idx.get("entailment", 1)

# Cache hop-1 max entailment scores once (threshold-agnostic)
rows = []
for r in tqdm(full_results, desc="Scoring hop-1 claims"):
    em = r["em"]
    answer = r["hop1_answer"]

    # Isolate sub-question 1 or default back to global main question
    q1 = r["sub_questions"][0] if "sub_questions" in r and len(r["sub_questions"]) >= 1 else r["question"]

     # Filter Strategy 1: Explicitly marked unanswerable scenarios
    if "UNANSWERABLE" in answer.upper():
        rows.append({"em": em, "state": "UNANSWERABLE", "max_ent": None})
        continue

    # Filter Strategy 2: Missing retrievals default to zero probability
    retrieved = r["retrieved_hop1"][:MAX_PASSAGES_VERIFY]
    if not retrieved:
        rows.append({"em": em, "state": "UNSUPPORTED", "max_ent": 0.0})
        continue

    # Transformation: Question-Answer statement flattening
    claim = qa_to_claim(q1, answer)
    ent_scores = []

    # Local NLI Batch loops over individual context passages
    for idx in retrieved:
        premise = corpus_docs[idx]
        inputs = nli_tokenizer(
            premise,
            claim,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True,
        ).to(DEVICE)

        with torch.no_grad():
            logits = nli_model(**inputs).logits

        # Extract soft probability distributions across NLI classes
        probs = torch.softmax(logits, dim=-1)[0]
        ent_scores.append(probs[ENTAILMENT_IDX].item())

    # Aggregate passage scores by picking maximum single-context entailment probability
    rows.append({"em": em, "state": "DECIDED", "max_ent": max(ent_scores) if ent_scores else 0.0})

# Sweep thresholds
THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]
summary = []

for t in THRESHOLDS:
    unsupported = []
    supported = []
    unanswerable = []

    # Loop over all cached NLI results to stratify them based on the current threshold
    for row in rows:
        # Route explicitly unanswerable samples into their own separate evaluation bucket
        if row["state"] == "UNANSWERABLE":
            unanswerable.append(row)
        else:
            # If the highest entailment score meets or exceeds the threshold,
            # consider the claim verified/supported by the retrieved context.
            if row["max_ent"] >= t:
                supported.append(row)
            # Otherwise, flag the claim as unsupported due to insufficient alignment.
            else:
                unsupported.append(row)

    # Track total dataset size and the subset of cases that weren't inherently unanswerable
    n = len(rows)
    decided_n = len(supported) + len(unsupported)

    # Calculate overall dataset proportions for unsupported and unanswerable samples.
    # Uses max(n, 1) to safely prevent ZeroDivisionError if rows is empty.
    unsupported_rate_all = len(unsupported) / max(n, 1)
    unanswerable_rate = len(unanswerable) / max(n, 1)


    # Quantify system error rates within each stratified group.
    # Evaluates failure rate by counting instances where downstream Exact Match (em) is 0.

     # Error rate among claims flagged as unsupported
    fail_if_unsupported = sum(1 for x in unsupported if x["em"] == 0) / max(len(unsupported), 1)
    # Error rate among claims flagged as supported
    fail_if_supported = sum(1 for x in supported if x["em"] == 0) / max(len(supported), 1)
    # Error rate among claims flagged as unanswerable
    fail_if_unanswerable = sum(1 for x in unanswerable if x["em"] == 0) / max(len(unanswerable), 1)

    # LIFT CALCULATION: Measures how effectively the verifier separates wrong answers from right ones.
    # Formula: Error rate when model says "Unsupported" MINUS Error rate when model says "Supported".
    # A higher, positive lift proves that the system is successfully isolating incorrect pipeline
    # behaviors (high errors in unsupported group) while validating reliable data (low errors in supported group).
    lift = fail_if_unsupported - fail_if_supported

    # Aggregate all calculated metrics for the current threshold into a
    # tracking list, which will be used for comparison and reporting.
    summary.append({
        "threshold": t,
        "unsupported_rate_all": unsupported_rate_all,
        "unanswerable_rate": unanswerable_rate,
        "fail_if_unsupported": fail_if_unsupported,
        "fail_if_supported": fail_if_supported,
        "fail_if_unanswerable": fail_if_unanswerable,
        "decided_n": decided_n,
        "lift": lift,
    })

# Print summary table
print("\n── Threshold Sweep Summary (hop 1) ──")
print(f"{'Thr':>5} {'Unsup%':>7} {'Unans%':>7} {'Fail|unsup':>11} {'Fail|sup':>9} {'Lift':>8} {'Decided n':>10}")
print("─" * 80)

# Scrape the tracking list to extract the optimal configuration dictionary based on maximum Lift
best = max(summary, key=lambda x: x["lift"])

# Iterate through every evaluated threshold configuration to construct table rows
for s in summary:
    # Append a visual indicator arrow if this row matches the absolute best threshold found
    marker = " ◄" if s["threshold"] == best["threshold"] else ""
    print(
        f"{s['threshold']:>5.2f} "
        f"{s['unsupported_rate_all']:>7.1%} "
        f"{s['unanswerable_rate']:>7.1%} "
        f"{s['fail_if_unsupported']:>11.1%} "
        f"{s['fail_if_supported']:>9.1%} "
        f"{s['lift']:>+8.3f} "
        f"{s['decided_n']:>10}{marker}"
    )

print(f"\nBest threshold by lift: {best['threshold']:.2f}")
print(f"Lift={best['lift']:+.3f} | Unsupported={best['unsupported_rate_all']:.1%} | UNANSWERABLE={best['unanswerable_rate']:.1%}")

No OpenAI API calls in this cell — local NLI inference only.

Running on: cuda
Verifier model: cross-encoder/nli-deberta-v3-base
Examples: 2,000
Per-passage aggregation over top 3



Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Scoring hop-1 claims:   0%|          | 0/2000 [00:00<?, ?it/s]


── Threshold Sweep Summary (hop 1) ──
  Thr  Unsup%  Unans%  Fail|unsup  Fail|sup     Lift  Decided n
────────────────────────────────────────────────────────────────────────────────
 0.30   31.8%   21.2%       52.4%     49.3%   +0.031       1575
 0.40   32.1%   21.2%       52.3%     49.3%   +0.030       1575
 0.50   32.6%   21.2%       52.4%     49.2%   +0.031       1575 ◄
 0.60   33.1%   21.2%       52.0%     49.5%   +0.026       1575
 0.70   33.6%   21.2%       51.7%     49.7%   +0.020       1575

Best threshold by lift: 0.50
Lift=+0.031 | Unsupported=32.6% | UNANSWERABLE=21.2%


In [3]:
# 7.2 Full grounding analysis (both hops) using BEST_VERIFIER
# Produces the Pipeline 2 faithfulness baseline file for Pipeline 3.

# Instantiate the optimized Natural Language Inference (NLI) verifier model selected from step 7.1
print(f"Loading winning verifier: {BEST_VERIFIER}...")
nli_tokenizer = AutoTokenizer.from_pretrained(BEST_VERIFIER)
nli_model = AutoModelForSequenceClassification.from_pretrained(BEST_VERIFIER)
nli_model.eval()
nli_model = nli_model.to(DEVICE)

# Map human-readable labels to internal class indices to find the exact 'entailment' target
lbl_map = {v.lower(): k for k, v in nli_model.config.id2label.items()}
ENTAILMENT_IDX = lbl_map.get("entailment", 1)
# Generate an execution wrapper closure for evaluating premises against claims
verify_answer = make_verifier(nli_tokenizer, nli_model, DEVICE, ENTAILMENT_IDX)

print(f"Loaded on {DEVICE}")
print(f"Running grounding checks on both hops ({len(full_results):,} examples)...")

# Evaluate correctness flags across individual reasoning layers (Hop 1 and Hop 2)
for r in tqdm(full_results, desc="NLI verification"):
    # Extract sub-questions for individual hops; default to the global main question if missing
    q1 = r["sub_questions"][0] if "sub_questions" in r and len(r["sub_questions"]) >= 1 else r["question"]
    q2 = r["sub_questions"][1] if "sub_questions" in r and len(r["sub_questions"]) >= 2 else r["question"]

    # Execute text-grounding validation using the generated NLI wrapper function
    r["hop1_verification"] = verify_answer(q1, r["hop1_answer"], r["retrieved_hop1"])
    r["hop2_verification"] = verify_answer(q2, r["hop2_answer"], r["retrieved_hop2"])

     # Flatten validation results into pure binary flags for cleaner group sorting later
    r["hop1_grounded"] = r["hop1_verification"]["supported"]
    r["hop2_grounded"] = r["hop2_verification"]["supported"]

print("Verification complete.\n")

# Identify per hop failure rates
n = len(full_results)
# Identify cases where the pipeline hallucinates or fails grounding check on a specific hop
hop1_ungrounded = [r for r in full_results if not r["hop1_grounded"]]
hop2_ungrounded = [r for r in full_results if not r["hop2_grounded"]]

print("── Per-hop verification failure rate ──")
print(f"  Hop 1 ungrounded: {len(hop1_ungrounded):>5}  ({len(hop1_ungrounded)/n:.1%})")
print(f"  Hop 2 ungrounded: {len(hop2_ungrounded):>5}  ({len(hop2_ungrounded)/n:.1%})")
print()

# Stratify rows into mutually exclusive buckets depending on overall grounding states
both_grounded = [r for r in full_results if r["hop1_grounded"] and r["hop2_grounded"]]
hop1_only_flagged = [r for r in full_results if not r["hop1_grounded"] and r["hop2_grounded"]]
hop2_only_flagged = [r for r in full_results if r["hop1_grounded"] and not r["hop2_grounded"]]
both_flagged = [r for r in full_results if not r["hop1_grounded"] and not r["hop2_grounded"]]
one_flagged = hop1_only_flagged + hop2_only_flagged

def mean_em(group):
    return sum(r["em"] for r in group) / len(group) if group else float("nan")

# Print structured table detailing accuracy across combinations of grounded states
print("── Conditional EM by grounding state ──")
print(f"  {'Group':<37} {'n':>5}  {'Mean EM':>8}")
print("  " + "─" * 54)
print(f"  {'Both grounded':<37} {len(both_grounded):>5}  {mean_em(both_grounded):>8.4f}")
print(f"  {'One flagged (combined)':<37} {len(one_flagged):>5}  {mean_em(one_flagged):>8.4f}")
print(f"    {'↳ Hop 1 only flagged':<35} {len(hop1_only_flagged):>5}  {mean_em(hop1_only_flagged):>8.4f}")
print(f"    {'↳ Hop 2 only flagged':<35} {len(hop2_only_flagged):>5}  {mean_em(hop2_only_flagged):>8.4f}")
print(f"  {'Both flagged':<37} {len(both_flagged):>5}  {mean_em(both_flagged):>8.4f}")
print()

# Extract samples that achieve correct global output (EM=1)
correct = [r for r in full_results if r["em"] == 1]
# Group correct samples by internal execution truthfulness
faithful_correct = [r for r in correct if r["hop1_grounded"] and r["hop2_grounded"]]
unfaithful_correct = [r for r in correct if not r["hop1_grounded"] or not r["hop2_grounded"]]

print("── Faithfulness failure rate ──")
print(f"  Total correct answers (EM=1):                {len(correct):>5}  ({len(correct)/n:.1%})")
print(f"  Faithful correct (both hops grounded):       {len(faithful_correct):>5}  ({len(faithful_correct)/max(len(correct),1):.1%} of correct)")
print(f"  Unfaithful correct (≥1 hop ungrounded):      {len(unfaithful_correct):>5}  ({len(unfaithful_correct)/max(len(correct),1):.1%} of correct)")
print()

# Partition results purely based on the foundational reasoning link (Hop 1)
hop1_grounded_group = [r for r in full_results if r["hop1_grounded"]]
hop1_ungrounded_group = [r for r in full_results if not r["hop1_grounded"]]
em_grounded = mean_em(hop1_grounded_group)
em_ungrounded = mean_em(hop1_ungrounded_group)

# PROPAGATION PENALTY: Quantifies drop in end-to-end downstream performance
# caused specifically by failing to establish factual grounding at step 1.
prop_penalty = em_grounded - em_ungrounded

print("── Propagation penalty ──")
print(f"  Mean EM when hop 1 grounded:     {em_grounded:.4f}  (n={len(hop1_grounded_group)})")
print(f"  Mean EM when hop 1 ungrounded:   {em_ungrounded:.4f}  (n={len(hop1_ungrounded_group)})")
print(f"  Propagation penalty:             {prop_penalty:+.4f}")
print()

# Load metadata safely in case BEST_MODE/BEST_TOP_K aren't in memory
with open(f"{SAVE_DIR}multihop_rag_results.json") as f:
    _meta = json.load(f)

# Fallback pattern matching: prioritize active runtime memory strings, otherwise pull from metadata
_best_mode = BEST_MODE if "BEST_MODE" in dir() else _meta.get("best_mode", "hybrid")
_best_top_k = BEST_TOP_K if "BEST_TOP_K" in dir() else _meta.get("best_top_k", 10)
_embed = EMBED_MODEL if "EMBED_MODEL" in dir() else _meta.get("embed_model", "all-MiniLM-L6-v2")
_metrics = full_metrics if "full_metrics" in dir() else _meta.get("metrics", {})

# Consolidate pipeline data, validation arrays, and configuration traces into one baseline payload
grounding_output = {
    "pipeline": "multihop_rag_with_grounding",
    "split": "shared_eval_2000_seed42",
    "n": n,
    "best_mode": _best_mode,
    "best_top_k": _best_top_k,
    "embed_model": _embed,
    "llm": "gpt-4o-mini",
    "nli_model": BEST_VERIFIER,
    "metrics": _metrics,
    "results": full_results,
}

# Write out baseline structure to Drive storage to initialize Pipeline 3 operations
with open(f"{SAVE_DIR}multihop_rag_grounding_results.json", "w") as f:
    json.dump(grounding_output, f, indent=2)

print(f"Saved → {SAVE_DIR}multihop_rag_grounding_results.json")
print(f"Verifier to carry forward into Pipeline 3: {BEST_VERIFIER}")

Loading winning verifier: cross-encoder/nli-deberta-v3-base...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loaded on cuda
Running grounding checks on both hops (2,000 examples)...


NLI verification:   0%|          | 0/2000 [00:00<?, ?it/s]

Verification complete.

── Per-hop verification failure rate ──
  Hop 1 ungrounded:  1078  (53.9%)
  Hop 2 ungrounded:  1336  (66.8%)

── Conditional EM by grounding state ──
  Group                                     n   Mean EM
  ──────────────────────────────────────────────────────
  Both grounded                           336    0.5982
  One flagged (combined)                  914    0.4453
    ↳ Hop 1 only flagged                  328    0.4268
    ↳ Hop 2 only flagged                  586    0.4556
  Both flagged                            750    0.3200

── Faithfulness failure rate ──
  Total correct answers (EM=1):                  848  (42.4%)
  Faithful correct (both hops grounded):         201  (23.7% of correct)
  Unfaithful correct (≥1 hop ungrounded):        647  (76.3% of correct)

── Propagation penalty ──
  Mean EM when hop 1 grounded:     0.5076  (n=922)
  Mean EM when hop 1 ungrounded:   0.3525  (n=1078)
  Propagation penalty:             +0.1551

Saved → /content/

In [4]:
# Final report summary cell
# Prints both: (1) P1 subset vs P2 comparison metrics, and (2) grounding metrics from 7.2.

import json
import numpy as np
from google.colab import drive

# Establish paths and connect to Google Drive if variables are not already in memory
if "SAVE_DIR" not in dir():
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"


# Computes baseline arithmetic averages for text generation accuracy,
# document retrieval coverage, and system-level response abstention rates.
def quick_summarize(results):
    n = max(len(results), 1)
    return {
        "EM": sum(r.get("em", 0) for r in results) / n,
        "F1": sum(r.get("f1", 0.0) for r in results) / n,
        "Recall@k (soft)": sum(r.get("recall_soft", 0) for r in results) / n,
        "Recall@k (strict)": sum(r.get("recall_strict", 0) for r in results) / n,
        "Abstention rate": sum(r.get("abstained", 0) for r in results) / n,
        "n": len(results),
    }


# Load Pipeline 1 full results + shared subset indices
with open(f"{SAVE_DIR}naive_rag_full_results.json") as f:
    p1_full_output = json.load(f)

# Load uniform array masks to slice identical indices across disparate runs
eval_indices_shared = np.load(f"{SAVE_DIR}eval_indices_shared.npy")
p1_full_results = p1_full_output["results"]

# Filter out matching evaluations, ensuring array slices stay safely within index boundaries
p1_subset_results = [p1_full_results[int(i)] for i in eval_indices_shared if int(i) < len(p1_full_results)]
p1_subset_metrics = quick_summarize(p1_subset_results)


# Load Pipeline 2 metrics
# Load Multi-Hop RAG (Pipeline 2) outputs
with open(f"{SAVE_DIR}multihop_rag_results.json") as f:
    p2_output = json.load(f)

# Retrieve cached metrics block; fall back to calculations if dictionary fields are missing
p2_metrics = p2_output.get("metrics", quick_summarize(p2_output.get("results", [])))


# Load grounding results from 7.2
# Pull structural data generated from the NLI verifier sweeps in module 7.2
with open(f"{SAVE_DIR}multihop_rag_grounding_results.json") as f:
    g_output = json.load(f)

g_results = g_output["results"]

# Recompute faithfulness metrics directly from saved grounding labels
n = len(g_results)

# Count individual hop failures where NLI validation scores fell under criteria thresholds
hop1_ungrounded = [r for r in g_results if not r.get("hop1_grounded", False)]
hop2_ungrounded = [r for r in g_results if not r.get("hop2_grounded", False)]

# Stratify rows into mutually exclusive evaluation groups based on dual-hop validation
both_grounded = [r for r in g_results if r.get("hop1_grounded", False) and r.get("hop2_grounded", False)]
hop1_only_flagged = [r for r in g_results if (not r.get("hop1_grounded", False)) and r.get("hop2_grounded", False)]
hop2_only_flagged = [r for r in g_results if r.get("hop1_grounded", False) and (not r.get("hop2_grounded", False))]
both_flagged = [r for r in g_results if (not r.get("hop1_grounded", False)) and (not r.get("hop2_grounded", False))]
one_flagged = hop1_only_flagged + hop2_only_flagged


def mean_em(group):
    return (sum(r.get("em", 0) for r in group) / len(group)) if group else float("nan")

# Identify instances that achieve correct output strings (EM=1)
correct = [r for r in g_results if r.get("em", 0) == 1]

# Differentiate true-positive reasoning from hallucinations ("Right for the wrong reasons")
faithful_correct = [r for r in correct if r.get("hop1_grounded", False) and r.get("hop2_grounded", False)]
unfaithful_correct = [r for r in correct if (not r.get("hop1_grounded", False)) or (not r.get("hop2_grounded", False))]

# Partition evaluation instances purely based on foundational sub-question verification
hop1_grounded_group = [r for r in g_results if r.get("hop1_grounded", False)]
hop1_ungrounded_group = [r for r in g_results if not r.get("hop1_grounded", False)]
em_grounded = mean_em(hop1_grounded_group)
em_ungrounded = mean_em(hop1_ungrounded_group)

# PROPAGATION PENALTY: Measures drop-off in accuracy caused by flawed upstream context loops
prop_penalty = em_grounded - em_ungrounded


# Print summary
print("══ Pipeline Comparison (P1 subset vs P2) ══")
print(f"{'Metric':<25} {'P1 subset':>12} {'P2':>12} {'Delta':>10}")
print("─" * 62)
for metric in ["EM", "F1", "Recall@k (soft)", "Recall@k (strict)", "Abstention rate"]:
    v1 = p1_subset_metrics[metric]
    v2 = p2_metrics[metric]
    d = v2 - v1
    arrow = "▲" if d > 0.001 else ("▼" if d < -0.001 else "─")
    print(f"{metric:<25} {v1:>12.4f} {v2:>12.4f} {arrow} {d:>+8.4f}")

# Print descriptive analytics summarizing system reliability metrics
print()
print("══ Grounding / Faithfulness Summary (from 7.2 output) ══")
print(f"Verifier: {g_output.get('nli_model', 'unknown')}")
print(f"n={n:,}")
print(f"Hop 1 ungrounded rate: {len(hop1_ungrounded)/max(n,1):.1%}")
print(f"Hop 2 ungrounded rate: {len(hop2_ungrounded)/max(n,1):.1%}")
print()
print("Conditional EM by grounding state:")
print(f"  Both grounded:        n={len(both_grounded):>4}  EM={mean_em(both_grounded):.4f}")
print(f"  One flagged combined: n={len(one_flagged):>4}  EM={mean_em(one_flagged):.4f}")
print(f"    - Hop1 only:        n={len(hop1_only_flagged):>4}  EM={mean_em(hop1_only_flagged):.4f}")
print(f"    - Hop2 only:        n={len(hop2_only_flagged):>4}  EM={mean_em(hop2_only_flagged):.4f}")
print(f"  Both flagged:         n={len(both_flagged):>4}  EM={mean_em(both_flagged):.4f}")
print()
print("Faithfulness failure:")
print(f"  Correct answers (EM=1):               {len(correct):>4} ({len(correct)/max(n,1):.1%})")
print(f"  Faithful correct (both grounded):      {len(faithful_correct):>4} ({len(faithful_correct)/max(len(correct),1):.1%} of correct)")
print(f"  Unfaithful correct (≥1 ungrounded):    {len(unfaithful_correct):>4} ({len(unfaithful_correct)/max(len(correct),1):.1%} of correct)")
print()
print("Propagation penalty:")
print(f"  EM when hop1 grounded:   {em_grounded:.4f} (n={len(hop1_grounded_group)})")
print(f"  EM when hop1 ungrounded: {em_ungrounded:.4f} (n={len(hop1_ungrounded_group)})")
print(f"  Penalty (grounded - ungrounded): {prop_penalty:+.4f}")

══ Pipeline Comparison (P1 subset vs P2) ══
Metric                       P1 subset           P2      Delta
──────────────────────────────────────────────────────────────
EM                              0.3385       0.4240 ▲  +0.0855
F1                              0.4532       0.5552 ▲  +0.1019
Recall@k (soft)                 0.9525       0.9570 ▲  +0.0045
Recall@k (strict)               0.4610       0.6950 ▲  +0.2340
Abstention rate                 0.3470       0.2400 ▼  -0.1070

══ Grounding / Faithfulness Summary (from 7.2 output) ══
Verifier: cross-encoder/nli-deberta-v3-base
n=2,000
Hop 1 ungrounded rate: 53.9%
Hop 2 ungrounded rate: 66.8%

Conditional EM by grounding state:
  Both grounded:        n= 336  EM=0.5982
  One flagged combined: n= 914  EM=0.4453
    - Hop1 only:        n= 328  EM=0.4268
    - Hop2 only:        n= 586  EM=0.4556
  Both flagged:         n= 750  EM=0.3200

Faithfulness failure:
  Correct answers (EM=1):                848 (42.4%)
  Faithful correct (both 